# Zip & Enumerate — Session 03: Pairing and Numbering

Two small functions, `zip()` and `enumerate()`, remove a huge amount of everyday Python pain. `zip()` glues several sequences together element by element, so you never have to index them in parallel. `enumerate()` gives you the position of each item while you loop, so you never have to write `range(len(...))` again.

**The big picture.** Real data arrives in parallel columns: names next to marks, product names next to prices, ages next to cities. `zip()` interleaves those columns into tuples, and `enumerate()` stamps each row with a running index. Together they turn clumsy index arithmetic into readable, Pythonic code.

**Why it matters.** Both return **lazy iterators**, so they inherit the single-use behaviour from the iterators notebook. They also power common idioms: `dict(zip(keys, values))` to build a dictionary, `zip(*rows)` to transpose a table, and `sorted(zip(values, labels), reverse=True)` to rank items.

**Roadmap.** In this notebook you will:
1. Pair two or more lists with `zip()` and see its lazy, single-use nature.
2. Handle uneven inputs, including the `strict=True` guard.
3. Use the unzip trick `zip(*zipped)` and the `dict(zip(...))` factory.
4. Add indices with `enumerate()`, including a custom `start=`.
5. Replace `range(len(...))` and hunt positions cleanly.
6. Build a student leaderboard from a real CSV.

**Key definitions.** *`zip(a, b)`*: a lazy iterator of tuples `(a_i, b_i)` that stops at the shortest input. *`enumerate(seq, start=0)`*: a lazy iterator of `(index, item)` pairs. *Unzip*: `zip(*pairs)` separates a list of tuples back into parallel tuples.

## 1. zip() — Interlocking the Teeth

`zip()` takes two or more iterables and produces tuples that pair their elements by position: the first items together, the second items together, and so on. It is named after a physical zipper that interleaves two rows of teeth into one track.

Like all the tools in this section, `zip` is **lazy** and returns an iterator, not a list. Iterate it once and it is spent; a second pass gives an empty result. Wrap it in `list()` or `dict()` to materialise it, and remember that materialising consumes it.

```text
names = ['Sana', 'Vikram', 'Priya']
marks = [89, 76, 95]
list(zip(names, marks))  ->  [('Sana', 89), ('Vikram', 76), ('Priya', 95)]
```

`zip` accepts any number of iterables, so `zip(names, marks, cities)` yields three-element tuples. The most common production uses are building dictionaries and ranking lists, both shown later in this notebook.

In [1]:
# 🐣 ZIP basics — teeth interlock!
names  = ["Sana", "Vikram", "Priya"]
marks  = [89, 76, 95]
pairs = zip(names, marks)
print("zip type :", type(pairs).__name__, "(L-A-G lazy hai!)")
print("list()   :", list(pairs))
print("dubara   :", list(pairs), "← ek-heart, dobara khaali! (nb1 wala trap!)")

# 3 lists bhi zipper-te:
cities = ["Delhi", "Pune", "Jaipur"]
print("\n3-zip  :", list(zip(names, marks, cities)))
# Expected OUTPUT:
# zip type : zip (L-A-G lazy hai!)
# list()   : [('Sana', 89), ('Vikram', 76), ('Priya', 95)]
# dubara   : [] ← ek-heart, dobara khaali! (nb1 wala trap!)
#
# 3-zip  : [('Sana', 89, 'Delhi'), ('Vikram', 76, 'Pune'), ('Priya', 95, 'Jaipur')]

zip type : zip (L-A-G lazy hai!)
list()   : [('Sana', 89), ('Vikram', 76), ('Priya', 95)]
dubara   : [] ← ek-heart, dobara khaali! (nb1 wala trap!)

3-zip  : [('Sana', 89, 'Delhi'), ('Vikram', 76, 'Pune'), ('Priya', 95, 'Jaipur')]


## 2. Uneven Inputs, strict=True, and the Unzip Trick

When the inputs differ in length, `zip` follows the **shortest-list rule**: it stops as soon as any input is exhausted, silently discarding the extras. That is usually convenient, but it can hide a data bug — a column with missing rows quietly truncates the join.

Python 3.10 added `strict=True` as a guard: if the lengths differ, `zip` raises `ValueError` instead of truncating. Use it when the columns are *supposed* to be equal, so mismatches fail loudly. When you genuinely want to keep the longest input, `itertools.zip_longest` fills missing slots with a default.

The reverse operation is the **unzip trick**. Because `zip(*rows)` unpacks a list of tuples into separate arguments, it transposes them:

```text
rows = [('x', 1), ('y', 2), ('z', 3)]
letters, digits = zip(*rows)   # ('x','y','z'), (1,2,3)
```

This one-liner turns row-oriented data back into columns, and is the backbone of transposing tables and rebuilding dictionaries.

In [2]:
# 🐥 UNEVEN lists + strict guard! (3.10+)
short = [1, 2]
long  = ["a", "b", "c", "d"]
print("normal zip   :", list(zip(short, long)), "← shortest-ka aangam!")

try:
    list(zip(short, long, strict=True))
except ValueError as e:
    print("strict=True 🚨:", e)

# unzip-trick: zip ka reverse? star-se kholo! ⭐
zipped = [("x", 1), ("y", 2), ("z", 3)]
letters, digits = zip(*zipped)
print("\nunzip letters:", letters)
print("unzip digits :", digits)
# Expected OUTPUT:
# normal zip   : [(1, 'a'), (2, 'b')] ← shortest-ka aangam!
# strict=True 🚨: zip() argument 2 is shorter than argument 1
#
# unzip letters: ('x', 'y', 'z')
# unzip digits : (1, 2, 3)

normal zip   : [(1, 'a'), (2, 'b')] ← shortest-ka aangam!
strict=True 🚨: zip() argument 2 is longer than argument 1

unzip letters: ('x', 'y', 'z')
unzip digits : (1, 2, 3)


## 3. dict(zip(...)) — The Mini Dictionary Factory

Because `zip` yields `(key, value)` pairs, feeding it to `dict()` builds a mapping in a single expression: `dict(zip(airports, cities))`. This is the fastest way to combine two parallel lists into a lookup table, and it reads almost like English.

Pair it with the unzip trick and you can invert a dictionary. `zip(d.values(), d.keys())` produces swapped pairs, and `dict()` turns them into a reversed mapping. This works because dictionary keys are unique; if two original values are equal, the reversal keeps only the last one.

| Expression | Result |
|---|---|
| `dict(zip(keys, values))` | A dictionary from two parallel lists |
| `dict(zip(d.values(), d.keys()))` | The same dictionary with keys and values swapped |

Both forms are lazy on the `zip` side and materialise only when `dict()` consumes the pairs.

In [3]:
# 🐥 dict(zip(k, v)) — 10-second mini-dict factory! 📒
airports = ["DEL", "BOM", "PNQ"]
cities   = ["Delhi", "Mumbai", "Pune"]
air_map = dict(zip(airports, cities))
print("factory dict :", air_map)

# KEYS-VALUES ko hi swap karna ho? — unzip rezip ID se bolo!
reverse = dict(zip(air_map.values(), air_map.keys()))
print("double-zipped:", reverse)
# Expected OUTPUT:
# factory dict : {'DEL': 'Delhi', 'BOM': 'Mumbai', 'PNQ': 'Pune'}
# double-zipped: {'Delhi': 'DEL', 'Mumbai': 'BOM', 'Pune': 'PNQ'}

factory dict : {'DEL': 'Delhi', 'BOM': 'Mumbai', 'PNQ': 'Pune'}
double-zipped: {'Delhi': 'DEL', 'Mumbai': 'BOM', 'Pune': 'PNQ'}


## 4. enumerate() — The Queue-Token Machine

`enumerate(seq, start=0)` yields `(index, item)` pairs as it walks a sequence. It removes the classic `for i in range(len(seq))` pattern, where you fetch the value with `seq[i]`. With `enumerate` you get the index and the value together, which is clearer and less error-prone.

The `start` argument controls the first index — `start=1` gives human-friendly numbering, and any integer works. It is normally written as a keyword for readability.

A common use is finding positions: `[i for i, n in enumerate(names) if n == 'A']` returns every index where the value matches, without maintaining a counter by hand. Because `enumerate` returns a lazy iterator, wrap it in `list()` when you want to inspect it directly in a REPL or print statement.

In [4]:
# 🐣 ENUMERATE — queue-token machine! 🎫
items = ["chai", "samosa", "lassi"]
print("default (0-start):")
for i, item in enumerate(items):
    print(f"  token#{i} → {item}")

print("\nstart=1 (human-style):")
for i, item in enumerate(items, start=1):
    print(f"  token#{i} → {item}")
# Expected OUTPUT:
# default (0-start):
#   token#0 → chai
#   token#1 → samosa
#   token#2 → lassi
#
# start=1 (human-style):
#   token#1 → chai
#   token#2 → samosa
#   token#3 → lassi

default (0-start):
  token#0 → chai
  token#1 → samosa
  token#2 → lassi

start=1 (human-style):
  token#1 → chai
  token#2 → samosa
  token#3 → lassi


## 5. enumerate vs range(len)

Both loops below produce identical output, but they send very different signals:

```text
range(len(menu)) style : [f'{i}:{menu[i]}' for i in range(len(menu))]
enumerate style        : [f'{i}:{x}' for i, x in enumerate(menu)]
```

The first reaches for an index only to look the value back up; the second reads the value directly while carrying its position. The `enumerate` form is shorter, avoids off-by-one mistakes, and works on any iterable — including ones that have no `len()` or indexing, such as generators and file objects.

As a rule of thumb: if you ever type `range(len(...))`, stop and ask whether `enumerate` fits. It usually does. The only time you need a raw index range is when the index itself is the point and there is no sequence to walk.

In [5]:
# 🐔 enumerate vs range(len) — WHY FORMER GOAT 🐐
menu = ["idli", "dosa", "vada"]
print("C-style    :", [f"{i}:{menu[i]}" for i in range(len(menu))], "← old-school, clumsy")
print("Pythonic   :", [f"{i}:{x}" for i, x in enumerate(menu)], "← clean, furs NUMBERS+ITEMS saath!")

# enumerate with SENTINEL use — positions dhundna:
names = ["A", "B", "A", "A"]
hits = [i for i, n in enumerate(names) if n == "A"]
print("A-positions  :", hits, "← index WITHOUT tracking-by-hand!")
# Expected OUTPUT:
# C-style    : ['0:idli', '1:dosa', '2:vada'] ← old-school, clumsy
# Pythonic   : ['0:idli', '1:dosa', '2:vada'] ← clean, furs NUMBERS+ITEMS saath!
# A-positions  : [0, 2, 3] ← index WITHOUT tracking-by-hand!

C-style    : ['0:idli', '1:dosa', '2:vada'] ← old-school, clumsy
Pythonic   : ['0:idli', '1:dosa', '2:vada'] ← clean, furs NUMBERS+ITEMS saath!
A-positions  : [0, 2, 3] ← index WITHOUT tracking-by-hand!


## 6. Real Dataset — The Student Leaderboard

This section applies both tools to `students_marks.csv` (50 students). After loading the rows, it builds two parallel lists: student names and their totals across five subjects.

`zip(names, totals)` pairs the columns, and `sorted(zip(totals, names), reverse=True)` turns them into a ranking. Placing the total first is deliberate: `sorted` compares tuples from the first element, so the highest total ranks first, and names break ties alphabetically.

Finally, `enumerate(ranked[:5], start=1)` adds human-friendly rank numbers. The combination — zip, sort, enumerate — is the standard recipe for any leaderboard, and you will reuse it whenever order matters.

In [6]:
# 💾 REAL DATASET — Students Leaderboard: zip + enumerate ka combo! 🎓
import csv

def load_students():
    for path in ["../../00_Resources/datasets/students_marks.csv",
                 "00_Python/00_Resources/datasets/students_marks.csv"]:
        try:
            with open(path, newline="", encoding="utf-8") as f:
                return list(csv.DictReader(f))
        except FileNotFoundError:
            continue
    raise FileNotFoundError("students_marks.csv nahi mila!")

students = load_students()
subjects = ["math", "science", "english", "hindi", "computer"]

names   = [s["name"] for s in students]
totals  = [sum(int(s[c]) for c in subjects) for s in students]
print("zip first-3 :", list(zip(names, totals))[:3])

# Leaderboard — sort DESC, enumerate RANK!
ranked = sorted(zip(totals, names), reverse=True)
print("\n🏆 CLASS TOP-5:")
for rank, (t, n) in enumerate(ranked[:5], start=1):
    print(f"  #{rank} {n:<12} total={t}")
# Expected OUTPUT:
# zip first-3 : [('Sneha_1', 289), ('Pooja_2', 249), ('Arjun_3', 348)]
#
# 🏆 CLASS TOP-5:
#   #1 Manish_29    total=418
#   #2 Ananya_20    total=401
#   #3 Arjun_11     total=397
#   #4 Sneha_24     total=393
#   #5 Priya_44     total=390

FileNotFoundError: students_marks.csv nahi mila!

## ⚠️ 5 Common Mistakes (Zip/Enumerate Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Iterating `zip` again (silent-empty!) | Fresh `zip(...)` each pass — regen is cheap! |
| 2 | `enumerate(items, 1)` — forgetting the second arg's position | Put faith in the `start=1` keyword! ✔️ both work, keyword is readable |
| 3 | Assuming "all pairs formed" on an uneven zip | Shortest-stops rule! Need to save data? `strict=True` or zip_longest |
| 4 | `for i in range(len(lst))` in Python | `for i, x in enumerate(lst)` — Pythonic! |
| 5 | Confusion about the sort key of `sorted(zip(a,b))` | `sorted(zip(totals, names))` sorts the FIRST element (totals!) — the tuple-first rule |

> 🎯 **Tip:** in `sorted(zip(totals, names), reverse=True)` the tuple's first item
> decides the order — ranks come from totals, ties break by names (A-Z). Rock-solid trick! 💪

## Summary

- `zip(a, b)` lazily pairs elements and stops at the shortest input; `strict=True` turns mismatches into errors.
- `zip(*rows)` unzips a list of tuples back into columns.
- `dict(zip(keys, values))` builds a lookup table in one line, and the same trick reverses a dictionary.
- `enumerate(seq, start=n)` yields `(index, item)` pairs and replaces `range(len(...))`.
- Both tools return single-use lazy iterators, so materialise them with `list()` when you need to reuse the result.
- Real data work — pairing names with totals and ranking with `enumerate` — becomes short and readable.